In [1]:
from config import *
from data.uniner_utils import load_uniner_dataset
from data.adapters import uniner_to_gliner
from data.chunking import chunk_gliner_dataset
from data.split_utils import split_dataset
from data.crossner_utils import read_bio, build_gliner_dataset
from training.train_utils import load_gliner_model, train_gliner_model, evaluate_finetune_gliner_model

import torch

Gliner2 CrossNER datasets

In [2]:
import logging

logging.getLogger("transformers").setLevel(logging.ERROR)

for domain in ['ai', 'literature', 'music', 'politics', 'science']:
    model = load_gliner_model()

    train_tokens, train_labels = read_bio(f"{CROSSNER_PATH}/{domain}/train.txt")
    dev_tokens, dev_labels = read_bio(f"{CROSSNER_PATH}/{domain}/dev.txt")
    test_tokens, test_labels = read_bio(f"{CROSSNER_PATH}/{domain}/test.txt")
    
    train_crossner_data = build_gliner_dataset(train_tokens,train_labels)
    dev_crossner_data   = build_gliner_dataset(dev_tokens,dev_labels)
    test_crossner_data  = build_gliner_dataset(test_tokens,test_labels)

    print(f"Evaluating on \"{domain.upper()}\" domain:")
    
    evaluate_finetune_gliner_model(
        model=model,
        domain=domain,
        test_data=test_crossner_data,
        train_data=train_crossner_data,
        eval_data=dev_crossner_data
    )

Evaluating on "AI" domain:
P: 72.21%	R: 52.29%	F1: 60.66% Base Model
P: 78.91%	R: 73.63%	F1: 76.18% Fine Tuned Model
Evaluating on "LITERATURE" domain:
P: 79.16%	R: 65.23%	F1: 71.52% Base Model
P: 81.99%	R: 77.14%	F1: 79.49% Fine Tuned Model
Evaluating on "MUSIC" domain:
P: 77.02%	R: 76.98%	F1: 77.00% Base Model
P: 83.79%	R: 81.18%	F1: 82.46% Fine Tuned Model
Evaluating on "POLITICS" domain:
P: 66.36%	R: 75.08%	F1: 70.45% Base Model
P: 84.84%	R: 83.92%	F1: 84.38% Fine Tuned Model
Evaluating on "SCIENCE" domain:
P: 77.99%	R: 67.66%	F1: 72.46% Base Model
P: 80.47%	R: 78.05%	F1: 79.24% Fine Tuned Model


Gliner1 CrossNER Datasets

In [2]:
import logging

logging.getLogger("transformers").setLevel(logging.ERROR)

for domain in ['ai', 'literature', 'music', 'politics', 'science']:
    model = load_gliner_model(
        model_name = "urchade/gliner_small-v1"
    )

    train_tokens, train_labels = read_bio(f"{CROSSNER_PATH}/{domain}/train.txt")
    dev_tokens, dev_labels = read_bio(f"{CROSSNER_PATH}/{domain}/dev.txt")
    test_tokens, test_labels = read_bio(f"{CROSSNER_PATH}/{domain}/test.txt")
    
    train_crossner_data = build_gliner_dataset(train_tokens,train_labels)
    dev_crossner_data   = build_gliner_dataset(dev_tokens,dev_labels)
    test_crossner_data  = build_gliner_dataset(test_tokens,test_labels)

    print(f"Evaluating on \"{domain.upper()}\" domain:")
    
    evaluate_finetune_gliner_model(
        model=model,
        domain=domain,
        test_data=test_crossner_data,
        train_data=train_crossner_data,
        eval_data=dev_crossner_data
    )

Evaluating on "AI" domain:
Train Time: 99.62s
P: 68.08%	R: 64.73%	F1: 66.36% Base Model
P: 80.01%	R: 73.24%	F1: 76.48% Fine Tuned Model
Evaluating on "LITERATURE" domain:
Train Time: 108.21s
P: 74.76%	R: 68.62%	F1: 71.56% Base Model
P: 81.16%	R: 74.89%	F1: 77.90% Fine Tuned Model
Evaluating on "MUSIC" domain:
Train Time: 95.02s
P: 75.64%	R: 78.57%	F1: 77.08% Base Model
P: 84.82%	R: 82.91%	F1: 83.86% Fine Tuned Model
Evaluating on "POLITICS" domain:
Train Time: 127.05s
P: 66.19%	R: 77.81%	F1: 71.53% Base Model
P: 86.05%	R: 83.08%	F1: 84.54% Fine Tuned Model
Evaluating on "SCIENCE" domain:
Train Time: 98.42s
P: 69.74%	R: 64.68%	F1: 67.11% Base Model
P: 81.13%	R: 77.37%	F1: 79.20% Fine Tuned Model


In [2]:
# 1. Load UniNER dataset
uniner_dataset = load_uniner_dataset()
model = load_gliner_model()
tokenizer = model.data_processor.transformer_tokenizer

In [10]:
total = sum(p.numel() for p in model.parameters())
print(f"{total/1e6:.2f}M parameters")

152.65M parameters


In [2]:
# 2. Split (70/20/10 por defecto)
train_uniner_data, val_uniner_data, test_uniner_data = split_dataset(uniner_dataset)

# 3. Adapt UniNER → GLiNER format
train_uniner_data = uniner_to_gliner(train_uniner_data, tokenizer)
val_uniner_data   = uniner_to_gliner(val_uniner_data, tokenizer)
test_uniner_data  = uniner_to_gliner(test_uniner_data, tokenizer)

# 4. Chunking (512 tokens)
train_uniner_data = chunk_gliner_dataset(train_uniner_data, max_length=512)
val_uniner_data   = chunk_gliner_dataset(val_uniner_data, max_length=512)
test_uniner_data  = chunk_gliner_dataset(test_uniner_data, max_length=512)

Eval Base Model

In [ ]:
model.eval()
with torch.no_grad():
    metrics = model.evaluate(
        test_uniner_data,
        batch_size=8
    )

print(metrics)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


('P: 55.50%\tR: 31.18%\tF1: 39.93%\n', np.float64(0.3993206440612747))


Train Model

In [ ]:
# 5. Train
train_gliner_model(
    model=model,
    train_data=train_uniner_data,
    eval_data=val_uniner_data,
    output_dir="./models/gliner_uniner",
    max_steps=100
)

NameError: name 'train_uniner_data' is not defined

Eval Trained Model

In [ ]:
model = load_gliner_model("./models/gliner_uniner/checkpoint-100")
model.eval()

with torch.no_grad():
    metrics = model.evaluate(
        test_uniner_data,
        batch_size=8
    )

print(metrics)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


('P: 66.13%\tR: 33.92%\tF1: 44.84%\n', np.float64(0.44837127749924666))


CrossNER

In [ ]:
import logging

logging.getLogger("transformers").setLevel(logging.ERROR)

for domain in ['ai', 'literature', 'music', 'politics', 'science']:
    model = load_gliner_model()

    train_tokens, train_labels = read_bio(f"{CROSSNER_PATH}/{domain}/train.txt")
    dev_tokens, dev_labels = read_bio(f"{CROSSNER_PATH}/{domain}/dev.txt")
    test_tokens, test_labels = read_bio(f"{CROSSNER_PATH}/{domain}/test.txt")
    
    train_crossner_data = build_gliner_dataset(train_tokens,train_labels)
    dev_crossner_data   = build_gliner_dataset(dev_tokens,dev_labels)
    test_crossner_data  = build_gliner_dataset(test_tokens,test_labels)

    print(f"Evaluating on \"{domain.upper()}\" domain:")
    
    evaluate_finetune_gliner_model(
        model=model,
        domain=domain,
        test_data=test_crossner_data,
        train_data=train_crossner_data,
        eval_data=dev_crossner_data
    )

Evaluating on "AI" domain:
{'train_runtime': 12.153, 'train_samples_per_second': 40.813, 'train_steps_per_second': 5.102, 'train_loss': 47.27436680947581, 'epoch': 4.769230769230769}
P: 72.78%	R: 52.18%	F1: 60.79% Base Model
P: 79.73%	R: 74.79%	F1: 77.18% Fine Tuned Model
Evaluating on "LITERATURE" domain:
{'train_runtime': 11.4793, 'train_samples_per_second': 43.208, 'train_steps_per_second': 5.401, 'train_loss': 52.015235162550404, 'epoch': 4.769230769230769}
P: 78.61%	R: 65.05%	F1: 71.19% Base Model
P: 82.23%	R: 75.55%	F1: 78.75% Fine Tuned Model
Evaluating on "MUSIC" domain:
{'train_runtime': 9.3497, 'train_samples_per_second': 53.05, 'train_steps_per_second': 6.631, 'train_loss': 43.120873235887096, 'epoch': 4.769230769230769}
P: 77.74%	R: 77.16%	F1: 77.45% Base Model
P: 83.96%	R: 82.55%	F1: 83.25% Fine Tuned Model
Evaluating on "POLITICS" domain:
{'train_runtime': 19.4093, 'train_samples_per_second': 51.522, 'train_steps_per_second': 6.44, 'train_loss': 37.18280078125, 'epoch': 5